In [1]:
import torch
from functools import partial
from typing import List

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

We can understand layouts by writing an elementwise add kernel, and this will help us quite a lot. 

Here is the rough idea: A,B,C are 2d tensors of shape (M,N)

indeed there is very little use of shared memory as the kernel is memory bound, so we will just think about gmem->reg->compute->gmem. 

first, our layouts will be the same across A,B,C (there is not reason not to) lets call the layout L = (M,N):(..,..) 

now, indeed, each block is going to process a tile of shape tile_shape = (block_M, block_N), and therefore we can zipped divide L with tile_shape to get a new layout L_TILED = ((tile_mode), (block_idx_mode)):(..,..) 
that is, that L_block_tile(block_idx) = L_TILED[(NONE), (block_idx)] is the tile of L that the block of block_idx is gonna work on. 

indeed, fixing a block_idx = b 

now a block is working on the tensor induced by L_block_tile(b) (as usual slicing adds offset to the engine start so to speak) 

now, we may want each thread to do some vectorized looading or whatever, that is, we may not want a thread to directly map to an element of L_block_tile(b) but rather another subtile of L_block_tile(b). This can be achieved using thread_value layouts or TV_layouts

We already know that layouts can go from coords-coords and not just coords to natural number offsets


Indeed, a TV_layout then best expressed as something where we can use threadidx to slice into our L_block_tile. 

But this can be decomposed as 2 independent parts: the act of slicing A co-ordinate layout, whose co-domain is the co-ordinate set of L_block_tile(b) by thread index, and composing that slicer layout with L_block_tile(b) itself.


what I mean is:  A TV layout is a rank-2 layout (its got 2 modes) where:

TV_layout = ((thread_mode),(value_mode)):((...),(....)) 

Indeed, due to the multiple ways of accessing layouts, there are many ways to think about this but one wayt that I like is that: TV_layout is a map ((linear_thread_idx),(value_coordinate)) -> L_block_tile(b)_coordinate

right, and then indeed, the co-ordinate slice owned by a thread t is then TV_layout((t),(NONE)) \
we also see that L_block_tile(b) is a map block_tile_coord -> block_tile_tensor_offset 


thus, we can just compose: L_block_tile(b) o TV_layout and then get map ((linear_thread_idx),(value_coordinate)) -> block_tile_tensor_offset

and then slice along the linear_thread_idx mode. 

For the below kernel, our tensors gA,gB,gC will already have the layout L_TILED = ((tile_mode), (block_idx_mode)):(..,..)  that is, we're gonna do it host side.

In [24]:
@cute.kernel
def elementwise_add_tv(gA:cute.Tensor, gB:cute.Tensor, gC:cute.Tensor, tv_layout: cute.Layout):
  tidx, _, _ = cute.arch.thread_idx()
  bidx, _, _ = cute.arch.block_idx()
  #we know that  gA,gB,gC have layout  L_TILED = ((tile_mode), (block_idx_mode)):(..,..)
  block_tile_slicer = (None,bidx) 
  blockA = gA[block_tile_slicer]
  blockB = gB[block_tile_slicer]
  blockC = gC[block_tile_slicer]
  #now since all these 3 share the same layout, we can compose it after tv_layout 
  #of course, we are actually composing a TENSOR with the tv layout, but a tesnosr T is just an engine after a layout 
  # T = (Engine o Layout) Hence T o TV_layout = Engine o (Layout o TV_layout) 
  #that is that the associative nature of function composition means that we just need to know how to compose layouts
  #and the composition of the engine against the resulting layout is equivalent ot adding the Engine iterator's start to the offsets produced
  
  thread_slicable_A = cute.composition(blockA, tv_layout)
  thread_slicable_B = cute.composition(blockB, tv_layout)  
  thread_slicable_C = cute.composition(blockC, tv_layout)
  
  thread_tile_slicer = (tidx,None)
  thread_tile_A = thread_slicable_A[thread_tile_slicer]
  thread_tile_B = thread_slicable_B[thread_tile_slicer]
  thread_tile_C = thread_slicable_C[thread_tile_slicer] 
  
  thread_tile_C[None] = thread_tile_A.load() + thread_tile_B.load() #tensorSSA abstraction I will go into it later. 
  
  

In [25]:
@cute.jit
def foo(): 
  coalesced_ldst_bytes = 16
  dtype = cutlass.BFloat16

  thr_layout = cute.make_ordered_layout((4, 64), order=(1, 0))
  cute.printf(f"thread_layout:{thr_layout}")
  val_layout = cute.make_ordered_layout((16, coalesced_ldst_bytes), order=(1, 0))
  cute.printf(f"value_layout:{val_layout}")
  val_layout = cute.recast_layout(dtype.width, 8, val_layout)
  cute.printf(f"value_layout_recast:{val_layout}")
  tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

  cute.printf(f"[DSL INFO] Tiler: {tiler_mn}")
  cute.printf(f"[DSL INFO] TV Layout: {tv_layout}")

In [26]:
foo()

thread_layout:(4,64):(64,1)
value_layout:(16,16):(16,1)
value_layout_recast:(16,8):(8,1)
[DSL INFO] Tiler: (64, 512)
[DSL INFO] TV Layout: ((64,4),(8,16)):((512,16),(64,1))


The above peice of code, depicts the idea of having one kernel, which can be launched with various layouts and this and that, normally when I write pure Cuda the kernel has fixed layout, sure sizes are tunable but layouts and access patterns are not. I mean I am not saying one cant write cute with tunable access patterns but if you wanna do all that elgantly I think you will rediscover some notions very close to layouts

So what is happening? the threadLayout is simply like okay we have 4*64 threads and we're gonna see it as if its (4,64) 2d and somehow its gonna be row major in the since that contigous threads are on the rows. 

Look the layouts (the math of it all) itself dont care if you think the offsets depict pointer offsets of an engine or linear thread indicies so its all fine. 

Now, the value layout is saying, that each thread, is going to work on a (16,16) row major tile, WHOSE ELEMENTS ARE BYTES. that is each thread is PRIMED to load 16 rows of 16 byte (128 bit) vectors, the TENSORSSA abstraction is gonna take care of that for us. (see the .load() in the kernel) 

but now, since we also want flexible dtypes, we are gonna cast the (16,16) row major byte granular tile to an element granular tile, assuming the dtype is some kind of float16, then the recast is obvious, 2 bytes is one element so our element granular recasted tile is gonna be of (16,8):(8,1)


of course, if one thread is gonna work on something of shape (16,8) and each block is (4,64) threads, then our tiler shape, is of (16*4, 64*8) = (64,512)

Now finally, there is one last thing running through our mind.

if TV Layouts are supposed to go 
((linear_thread_idx),(value_coordinate)) -> L_block_tile(b)_coordinate

then it must be a co-ordinate layout (we said that its gonna map coords to coords), therefore its strides must be nested tuples of scaled basis vectors. 

and yet, the above print clearly ((64,4),(8,16)):((512,16),(64,1)) gives us natural number strides, not scaled basis vectors.. hmmmmmmmmmmm 

okay, think about this.. WHY? like WHY, My best guess is that:

well look, the issue is, how are you going to compose two maps if the co-domain of the first one is different from the domain of the second one entirely? like not even subset, just completely disjoint. 

well you cant, which is exactly what would happen if TV_layouts were co-ordinate layouts in the sense of having the co-domain being co-ordinate tuples instead of 1d offsets

(recall that the domain of a layout is [0,L.size) which is a natural number domain) 

so it would not be possible for cute.composition, which I pressume is written for composing standard layouts only (not sure at this point)

Of course you might want to argue that hey just skip co-ordinate map of the TV_layout, and the colexical map of the layout we are comosing on the TV layout, and viola. but that, is kinda goofy 


but again, you can think of TV_layout as a standard layout but then also compose colex_inv_{TILER_M_N} to get co-ordinates, and indeed this would feed nicely into the layout we are compisng on the TV layout and also give us the internal satisfcation of thinking about TV layouts as giving out co-ordinates. 

All of this is under the assumption that the intention is that cute.make_tv_layout -> tv_layout, tiler_m_n is supposed to be sorta seen as actual_tv_layout ~ colex_inv_tiler_mn o tv_layout  


Okay fine lets just write the host side launcher now

In [43]:
M, N = 16384, 8192  
num_elements = M*N


In [44]:
@cute.jit
def elementwise_add(
    mA: cute.Tensor,
    mB: cute.Tensor,
    mC: cute.Tensor,
):

    coalesced_ldst_bytes = 16

    assert all(t.element_type == mA.element_type for t in [mA, mB, mC])
    dtype = mA.element_type

    thr_layout = cute.make_ordered_layout((4, 64), order=(1, 0))
    val_layout = cute.make_ordered_layout((16, coalesced_ldst_bytes), order=(1, 0))
    val_layout = cute.recast_layout(dtype.width, 8, val_layout)
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)

    print(f"[DSL INFO] Tiler: {tiler_mn}")
    print(f"[DSL INFO] TV Layout: {tv_layout}")

    gA = cute.zipped_divide(mA, tiler_mn)  
    gB = cute.zipped_divide(mB, tiler_mn)  
    gC = cute.zipped_divide(mC, tiler_mn)  # ((TileM, TileN), (RestM, RestN))

    print("Tiled Input Tensors:")
    print("[DSL INFO] Tiled Tensors:")
    print(f"[DSL INFO]   gA = {gA.type}")
    print(f"[DSL INFO]   gB = {gB.type}")
    print(f"[DSL INFO]   gC = {gC.type}")

    # Launch the kernel asynchronously
    # Async token(s) can also be specified as dependencies
    elementwise_add_tv(gA, gB, gC, tv_layout).launch(
        grid=[cute.size(gC, mode=[1]), 1, 1],
        block=[cute.size(tv_layout, mode=[0]), 1, 1],
    )


a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros(M, N, device="cuda", dtype=torch.float16)
print(c)
a_ = from_dlpack(a, assumed_align=16)
b_ = from_dlpack(b, assumed_align=16)
c_ = from_dlpack(c, assumed_align=16)

elementwise_add_ = cute.compile(elementwise_add, a_, b_, c_)
elementwise_add_(a_, b_, c_)

print(torch.testing.assert_close(c, a + b))
print(a+b)
print(c)

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0', dtype=torch.float16)
[DSL INFO] Tiler: (64, 512)
[DSL INFO] TV Layout: ((64,4),(8,16)):((512,16),(64,1))
Tiled Input Tensors:
[DSL INFO] Tiled Tensors:
[DSL INFO]   gA = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
[DSL INFO]   gB = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
[DSL INFO]   gC = !cute.memref<f16, gmem, align<16>, "((64,512),(256,16)):((8192,1),(524288,512))">
None
tensor([[-7.5488e-01, -1.9971e+00, -7.1826e-01,  ..., -7.5537e-01,
         -5.3906e-01,  2.0918e+00],
        [-4.4312e-01, -1.8184e+00, -5.4932e-01,  ...,  1.9531e-03,
          1.1660e+00, -1.3301e+00],
        [-2.2793e+00, -2.0645e+00,  1.7444e-01,  ...,  1.0645e+00,

In [45]:
def benchmark(callable, a_, b_, c_):
    avg_time_us = cute.testing.benchmark(
        callable,
        kernel_arguments=cute.testing.JitArguments(a_, b_, c_),
        warmup_iterations=5,
        iterations=100,
    )

    # Calculate metrics
    # ----------------
    dtype = a_.element_type

    # Calculate total bytes transferred:
    # - 2 reads (A and B) + 1 write (C)
    # - Each element is dtype.width bits
    bytes_per_element = dtype.width // 8
    total_bytes = num_elements * bytes_per_element

    # Calculate achieved bandwidth
    achieved_bandwidth = total_bytes / (avg_time_us * 1000)  # GB/s

    # Print results
    # ------------
    print(f"Performance Metrics:")
    print(f"-------------------")
    print(f"Kernel execution time: {avg_time_us:.4f} us")
    print(f"Memory throughput: {achieved_bandwidth:.2f} GB/s")

In [46]:
benchmark(elementwise_add_, a_, b_, c_)

Performance Metrics:
-------------------
Kernel execution time: 518.4781 us
Memory throughput: 517.74 GB/s
